# ShopFlow — Phase 1 Data Load
### Upload CSVs to stage → COPY INTO RAW tables → Verify

---

### Why no Kaggle download in this notebook?

Snowflake Notebooks run on Snowflake's managed compute. That runtime **cannot make outbound internet connections** to external sites like kaggle.com. Any attempt to call an external URL will fail with a connection error.

So we split the work:

```text
Your laptop                         Snowflake
──────────────────────────          ──────────────────────────────
Step A — Download CSVs              
         from kaggle.com            
                                    
Step B — Upload 9 CSV files  ─────► OLIST_STAGE (internal stage)
         via Snowsight UI           
                                    Step C — COPY INTO RAW tables
                                    Step D — Verify row counts
                                    Step E — Data quality checks
```

Steps A and B happen **before** you run this notebook. Steps C–E are what this notebook does.

---

### Before running this notebook — complete these two steps first

**Step A — Download the Olist dataset on your laptop**

Go to `kaggle.com/datasets/olistbr/brazilian-ecommerce` → click **Download**

Unzip the file. You should have these 9 CSV files:

| File | Size (approx) |
|------|---------------|
| olist_orders_dataset.csv | 12 MB |
| olist_order_items_dataset.csv | 9 MB |
| olist_customers_dataset.csv | 7 MB |
| olist_sellers_dataset.csv | 0.2 MB |
| olist_products_dataset.csv | 3 MB |
| olist_order_payments_dataset.csv | 5 MB |
| olist_order_reviews_dataset.csv | 55 MB |
| olist_geolocation_dataset.csv | 25 MB |
| product_category_name_translation.csv | 0.01 MB |

**Step B — Upload all 9 files to OLIST_STAGE via Snowsight UI**

1. In Snowsight left sidebar → **Data → Databases**
2. Navigate to `SHOPFLOW_DB → SHOPFLOW_RAW → Stages → OLIST_STAGE`
3. Click **+ Files** button (top right of the stage page)
4. Select all 9 CSV files → click **Upload**
5. Wait for all uploads to complete — Snowflake compresses them to `.gz` automatically

Once Step B is done, come back and run this notebook cell by cell.

---

### What this notebook does

| Cell | What happens |
|------|-------------|
| Cell 1 | Get Snowpark session · verify context |
| Cell 2 | Confirm all 9 files are visible in the stage |
| Cell 3 | COPY INTO all 9 RAW tables |
| Cell 4 | Verify row counts against expected values |
| Cell 5 | Check LOAD_HISTORY audit log |
| Cell 6 | Data quality spot checks |
| Cell 7 | Suspend warehouse + final summary |

---
## Cell 1 — Get Snowpark session and verify context

### Why `get_active_session()` and not `snowflake.connector.connect()`?

Inside a Snowflake Notebook you are already running on Snowflake's compute. The platform automatically injects a pre-authenticated session — no credentials, no account identifier, no password needed.

`session.sql('...').collect()` sends SQL to Snowflake's engine and returns results as a Python list of `Row` objects.

### What to check in the output

Confirm all four context values show the correct ShopFlow objects before proceeding:
- Warehouse → `SHOPFLOW_WH`
- Database → `SHOPFLOW_DB`
- Schema → `SHOPFLOW_RAW`
- Role → your role (e.g. `SYSADMIN` or `SHOPFLOW_ROLE`)

If any of these are wrong, the `USE` statements at the bottom of this cell will correct them.

In [ ]:
from snowflake.snowpark.context import get_active_session

# Get the pre-authenticated Snowpark session
# No credentials needed — Snowflake injects this automatically
session = get_active_session()

# Check current session context
ctx = session.sql("""
    SELECT
        CURRENT_USER()      AS current_user,
        CURRENT_ROLE()      AS current_role,
        CURRENT_WAREHOUSE() AS current_warehouse,
        CURRENT_DATABASE()  AS current_database,
        CURRENT_SCHEMA()    AS current_schema
""").collect()[0]

print(f"""
── Snowflake session context ───────────────────
  User      : {ctx['CURRENT_USER']}
  Role      : {ctx['CURRENT_ROLE']}
  Warehouse : {ctx['CURRENT_WAREHOUSE']}
  Database  : {ctx['CURRENT_DATABASE']}
  Schema    : {ctx['CURRENT_SCHEMA']}
────────────────────────────────────────────────
""")

# Point the session at the correct objects for this notebook
# These are safe to run even if context is already correct
# session.sql('USE WAREHOUSE SHOPFLOW_WH').collect()
session.sql('USE DATABASE SHOPFLOW_DB').collect()
session.sql('USE SCHEMA SHOPFLOW_RAW').collect()

print('✅ Session context set:')
print('   Warehouse → SHOPFLOW_WH')
print('   Database  → SHOPFLOW_DB')
print('   Schema    → SHOPFLOW_RAW')

---
## Cell 2 — Confirm files are in the stage

Before running COPY INTO, confirm that all 9 CSV files you uploaded via the Snowsight UI are visible in the stage.

### What is LIST?

`LIST @stage_name` shows all files currently sitting in a stage — their names, sizes, and last modified timestamps. It is the Snowflake equivalent of `ls` on a filesystem or `aws s3 ls` for an S3 bucket.

### What to look for in the output

You should see 9 files. Each file will have a `.gz` extension because Snowflake automatically compresses files when they are uploaded via the UI.

If you see fewer than 9 files — go back to Snowsight, open `OLIST_STAGE`, and upload the missing ones before continuing.

> **💡 Concept Note: UI Upload Compression**
> 
> When files are uploaded to an internal stage via the Snowsight UI, Snowflake compresses them to `.gz` automatically. When you reference files in `COPY INTO`, you reference the `.gz` filename — not the original `.csv` name. This is why `FILE_TABLE_MAP` below appends `.gz` to each filename.

In [ ]:
STAGE = '@SHOPFLOW_DB.SHOPFLOW_RAW.OLIST_STAGE'

# Master mapping: original CSV filename → RAW table name
# After UI upload, files are stored as filename.csv.gz in the stage
FILE_TABLE_MAP = {
    'olist_orders_dataset.csv'              : 'RAW_ORDERS',
    'olist_order_items_dataset.csv'         : 'RAW_ORDER_ITEMS',
    'olist_customers_dataset.csv'           : 'RAW_CUSTOMERS',
    'olist_sellers_dataset.csv'             : 'RAW_SELLERS',
    'olist_products_dataset.csv'            : 'RAW_PRODUCTS',
    'olist_order_payments_dataset.csv'      : 'RAW_PAYMENTS',
    'olist_order_reviews_dataset.csv'       : 'RAW_REVIEWS',
    'olist_geolocation_dataset.csv'         : 'RAW_GEOLOCATION',
    'product_category_name_translation.csv' : 'RAW_CATEGORY_TRANSLATION',
}

print(f'🔍 Checking stage contents: {STAGE}\n')

stage_files = session.sql(f'LIST {STAGE}').collect()

if not stage_files:
    print('❌ Stage is empty — no files found')
    print('   Go to Snowsight → SHOPFLOW_DB → SHOPFLOW_RAW → Stages → OLIST_STAGE')
    print('   Click + Files and upload all 9 CSV files before continuing')
else:
    print(f'   {"File name":<60} {"Size":>12}')
    print('   ' + '─' * 75)
    for row in stage_files:
        size_mb = int(row['size']) / (1024 * 1024)
        print(f"   {row['name']:<60} {size_mb:>9.1f} MB")

    print(f'\n   Files in stage : {len(stage_files)}')
    print(f'   Files expected : 9')

    if len(stage_files) == 9:
        print('\n✅ All 9 files present — ready to run COPY INTO')
    else:
        missing = 9 - len(stage_files)
        print(f'\n⚠️  {missing} file(s) missing — upload them before running Cell 3')

---
## Cell 3 — COPY INTO all 9 RAW tables

### What does COPY INTO do?

`COPY INTO` reads files from a stage and loads them into a Snowflake table. It is Snowflake's bulk loader — the equivalent of Redshift's `COPY` command.

```sql
COPY INTO target_table
FROM @stage/filename.csv.gz
FILE_FORMAT = (FORMAT_NAME = 'OLIST_CSV_FORMAT')
ON_ERROR    = 'SKIP_FILE'
PURGE       = FALSE
```

### Key options explained

**`FORMAT_NAME = 'OLIST_CSV_FORMAT'`** — references the file format created in the setup notebook. All CSV parsing rules (delimiter, quotes, NULLs, encoding) are defined there. One format, used by all 9 loads.

**`ON_ERROR = 'SKIP_FILE'`** — if a file has any parsing error, skip that file and continue loading the others. The load will not crash because of one bad file.

| ON_ERROR option | Behaviour |
|---|---|
| `ABORT_STATEMENT` | Default — any error stops everything immediately |
| `CONTINUE` | Skip bad rows, keep loading good rows from the same file |
| `SKIP_FILE` | Skip the entire file if any error found |
| `SKIP_FILE_n` | Skip file only if errors exceed n rows |

**`PURGE = FALSE`** — keep files in the stage after loading. Two reasons:
1. You can reload if something went wrong
2. Snowflake tracks which files have already been loaded — re-running COPY INTO on the same file **will not duplicate data** (Snowflake remembers load history per file)

> **🛡️ Reliability Tip: Idempotency**
> 
> `COPY INTO` is **idempotent by default**. If you run it twice on the exact same file, the second run will show status `SKIPPED` — Snowflake will not load the same file twice. To force a reload, add `FORCE = TRUE`.

In [ ]:
print('📥 Loading staged files into RAW tables via COPY INTO...\n')
print(f'   {"Table":<35} {"Rows":>10}   {"Errors":>7}   Status')
print('   ' + '─' * 65)

load_results = []

for filename, table_name in FILE_TABLE_MAP.items():

    # Files uploaded via Snowsight UI are stored as filename.csv.gz
    staged_file = filename + '.gz'
    full_table  = f'SHOPFLOW_DB.SHOPFLOW_RAW.{table_name}'

    copy_sql = f"""
        COPY INTO {full_table}
        FROM {STAGE}/{staged_file}
        FILE_FORMAT = (
            FORMAT_NAME = 'SHOPFLOW_DB.SHOPFLOW_RAW.OLIST_CSV_FORMAT'
        )
        ON_ERROR = 'SKIP_FILE'
        PURGE    = FALSE
    """

    try:
        result = session.sql(copy_sql).collect()

        if result:
            rows_loaded = result[0]['rows_loaded']
            errors      = result[0]['errors_seen']
            status      = result[0]['status']
        else:
            rows_loaded, errors, status = 0, 0, 'NO_RESULT'

        load_results.append({
            'table'      : table_name,
            'rows_loaded': rows_loaded,
            'errors'     : errors,
            'status'     : status
        })

        icon = '✅' if status in ('LOADED', 'SKIPPED') and errors == 0 else '⚠️'
        print(f'   {icon} {table_name:<33} {str(rows_loaded):>10}   {errors:>7}   {status}')

    except Exception as e:
        print(f'   ❌ {table_name}: {str(e)[:80]}')
        load_results.append({
            'table'      : table_name,
            'rows_loaded': 0,
            'errors'     : -1,
            'status'     : 'FAILED'
        })

# Summary
total_rows = sum(
    r['rows_loaded'] for r in load_results
    if isinstance(r['rows_loaded'], int)
)
failed = [r['table'] for r in load_results if r['status'] == 'FAILED']

print('   ' + '─' * 65)
print(f'\n   Total rows loaded  : {total_rows:,}')
print(f'   Tables succeeded   : {len(load_results) - len(failed)} / {len(FILE_TABLE_MAP)}')

if failed:
    print(f'\n   ⚠️  Failed tables: {failed}')
    print('   → Check Cell 5 (LOAD_HISTORY) for details')
else:
    print('\n✅ All tables loaded successfully')

---
## Cell 4 — Verify row counts

Never trust a load without verifying it. `COPY INTO` with `ON_ERROR = SKIP_FILE` can silently skip an entire file if it encounters a problem. A row count check costs nothing and catches any issues immediately.

### Expected counts

These are the documented row counts for the Olist dataset. If your actual counts match these, the load was successful.

A `SKIPPED` status in Cell 3 means the file was already loaded in a previous run — row counts will still be correct as long as the table is not empty.

In [ ]:
# Expected row counts from Olist dataset documentation
EXPECTED = {
    'RAW_ORDERS'              : 99441,
    'RAW_ORDER_ITEMS'         : 112650,
    'RAW_CUSTOMERS'           : 99441,
    'RAW_SELLERS'             : 3095,
    'RAW_PRODUCTS'            : 32951,
    'RAW_PAYMENTS'            : 103886,
    'RAW_REVIEWS'             : 99224,
    'RAW_GEOLOCATION'         : 1000163,
    'RAW_CATEGORY_TRANSLATION': 71,
}

print('🔍 Row count verification:\n')
print(f'   {"Table":<35} {"Actual":>10}  {"Expected":>10}  Result')
print('   ' + '─' * 72)

all_good = True

for table, exp in EXPECTED.items():
    actual = session.sql(
        f'SELECT COUNT(*) AS cnt FROM SHOPFLOW_DB.SHOPFLOW_RAW.{table}'
    ).collect()[0]['CNT']

    if actual == exp:
        result = '✅ match'
    elif actual == 0:
        result = '❌ empty — COPY INTO failed'
        all_good = False
    elif actual < exp:
        result = f'⚠️  low by {exp - actual:,}'
        all_good = False
    else:
        result = f'⚠️  high by {actual - exp:,}'
        all_good = False

    print(f'   {table:<35} {actual:>10,}  {exp:>10,}  {result}')

print('   ' + '─' * 72)

if all_good:
    print('\n✅ All 9 tables match expected row counts')
    print('   RAW layer load is complete — ready for Phase 2')
else:
    print('\n⚠️  Some counts do not match')
    print('   Check Cell 5 (LOAD_HISTORY) to see what went wrong')

---
## Cell 5 — Check LOAD_HISTORY

`INFORMATION_SCHEMA.LOAD_HISTORY` is a system view that records every `COPY INTO` operation run against tables in your database. It shows exactly what was loaded, when, how many rows, and whether there were errors.

### When to use this

- After any load — to confirm what actually happened
- When row counts don't match — to see error details
- When status shows `SKIPPED` — to confirm the file was loaded previously

> **🔍 Concept Note: LOAD_HISTORY vs COPY_HISTORY**
> 
> When tracking your data loads, you have two distinct views available. It is important to know which one to use:
> 
> | Feature | `INFORMATION_SCHEMA.LOAD_HISTORY` | `SNOWFLAKE.ACCOUNT_USAGE.COPY_HISTORY` |
> |---------|-----------------------------------|----------------------------------------|
> | **Scope** | Current database only | Entire account |
> | **Latency** | **Real-time** (Use for immediate checks) | Up to 45 minutes (Use for long-term reporting) |
> | **Retention** | 14 days | 365 days |
> | **Privilege needed** | Schema `USAGE` | `ACCOUNTADMIN` or `SNOWFLAKE` db privileges |

In [ ]:
import pandas as pd

print('🔍 LOAD_HISTORY — all COPY INTO operations on SHOPFLOW_RAW:\n')

history = session.sql("""
    SELECT
        TO_CHAR(LAST_LOAD_TIME, 'YYYY-MM-DD HH24:MI') AS load_time,
        TABLE_NAME,
        ROW_COUNT,
        ROW_PARSED,
        ERROR_COUNT,
        STATUS
    FROM SHOPFLOW_DB.INFORMATION_SCHEMA.LOAD_HISTORY
    WHERE SCHEMA_NAME = 'SHOPFLOW_RAW'
    ORDER BY LAST_LOAD_TIME DESC
    LIMIT 20
""").collect()

if history:
    df = pd.DataFrame([r.as_dict() for r in history])
    print(df.to_string(index=False))
    print(f'\n   Total operations shown: {len(history)}')
else:
    print('   No load history found yet')
    print('   Run Cell 3 first to load the data')

---
## Cell 6 — Data quality spot checks

The RAW layer stores data exactly as it arrived. Before moving to Phase 2 (STAGED), run these checks to understand what you are working with.

These checks answer five questions:
1. What is the order status mix?
2. What payment methods do customers use?
3. How satisfied are customers (review scores)?
4. Are there NULLs in critical columns?
5. Which product categories dominate?

### Note on VARCHAR casting

All RAW columns are `VARCHAR`. Any numeric operation requires an explicit cast:
```sql
AVG(payment_value::FLOAT)    -- cast to FLOAT for arithmetic
review_score::INTEGER        -- cast to INTEGER for correct ordering
```
This is intentional — it reminds you that you are working with raw strings, not typed data.

In [ ]:
import pandas as pd

print('🔍 Data quality snapshot\n')

checks = [
    {
        'name': '1. Order status distribution',
        'why' : 'Understand the delivered vs cancelled vs processing split',
        'sql' : """
            SELECT
                order_status,
                COUNT(*) AS order_count,
                ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
            FROM SHOPFLOW_DB.SHOPFLOW_RAW.RAW_ORDERS
            GROUP BY 1
            ORDER BY 2 DESC"""
    },
    {
        'name': '2. Payment type mix',
        'why' : 'Credit card vs boleto vs voucher — and average order value per type',
        'sql' : """
            SELECT
                payment_type,
                COUNT(*) AS payment_count,
                ROUND(AVG(payment_value::FLOAT), 2) AS avg_value_brl
            FROM SHOPFLOW_DB.SHOPFLOW_RAW.RAW_PAYMENTS
            GROUP BY 1
            ORDER BY 2 DESC"""
    },
    {
        'name': '3. Review score distribution',
        'why' : 'Customer satisfaction snapshot — 5-star vs 1-star ratio',
        'sql' : """
            SELECT
                review_score,
                COUNT(*) AS review_count,
                ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
            FROM SHOPFLOW_DB.SHOPFLOW_RAW.RAW_REVIEWS
            GROUP BY 1
            ORDER BY review_score::INTEGER"""
    },
    {
        'name': '4. NULL check on RAW_ORDERS key columns',
        'why' : 'Find data quality issues before STAGED type casting',
        'sql' : """
            SELECT
                COUNT(*)  AS total_rows,
                SUM(CASE WHEN order_id IS NULL
                         THEN 1 ELSE 0 END) AS null_order_id,
                SUM(CASE WHEN customer_id IS NULL
                         THEN 1 ELSE 0 END) AS null_customer_id,
                SUM(CASE WHEN order_status IS NULL
                         THEN 1 ELSE 0 END) AS null_status,
                SUM(CASE WHEN order_purchase_timestamp IS NULL
                         THEN 1 ELSE 0 END) AS null_purchase_ts,
                SUM(CASE WHEN order_delivered_customer_date IS NULL
                         THEN 1 ELSE 0 END) AS null_delivery_date
            FROM SHOPFLOW_DB.SHOPFLOW_RAW.RAW_ORDERS"""
    },
    {
        'name': '5. Top 10 product categories (English)',
        'why' : 'Which categories dominate — useful for Phase 3 dimension design',
        'sql' : """
            SELECT
                COALESCE(t.product_category_name_english, 'uncategorised') AS category_english,
                COUNT(*) AS product_count
            FROM SHOPFLOW_DB.SHOPFLOW_RAW.RAW_PRODUCTS p
            LEFT JOIN SHOPFLOW_DB.SHOPFLOW_RAW.RAW_CATEGORY_TRANSLATION t
                   ON p.product_category_name = t.product_category_name
            GROUP BY 1
            ORDER BY 2 DESC
            LIMIT 10"""
    }
]

for check in checks:
    print(f"── {check['name']} {'─' * max(1, 54 - len(check['name']))}")
    print(f"   Why: {check['why']}\n")
    rows = session.sql(check['sql']).collect()
    df   = pd.DataFrame([r.as_dict() for r in rows])
    print(df.to_string(index=False))
    print()

---
## Cell 7 — Suspend warehouse and final summary

### Why suspend manually?

`SHOPFLOW_WH` has `AUTO_SUSPEND = 60` so it will suspend itself after 60 seconds of idle time anyway. But explicit suspension is good practice:

- Stops the billing clock immediately — no waiting for the 60-second idle timer
- In production pipelines, always end a job with an explicit suspend

> **💸 Cost Optimization Tip: Warehouse Billing**
> 
> Snowflake warehouses are billed in **60-second minimum increments** every time they start. A query that takes 3 seconds still consumes a full 60-second credit block. Having `AUTO_SUSPEND = 60` means there is at most one extra 60-second block of idle cost after your last query finishes. Manually suspending the warehouse as soon as your job finishes eliminates even that extra minute of waste!

In [ ]:
# Suspend immediately — do not wait for AUTO_SUSPEND
session.sql('ALTER WAREHOUSE SHOPFLOW_WH SUSPEND').collect()
print('✅ SHOPFLOW_WH suspended — credits saved')

# Grand total across all 9 tables
total = session.sql("""
    SELECT
        (SELECT COUNT(*) FROM SHOPFLOW_DB.SHOPFLOW_RAW.RAW_ORDERS)               +
        (SELECT COUNT(*) FROM SHOPFLOW_DB.SHOPFLOW_RAW.RAW_ORDER_ITEMS)          +
        (SELECT COUNT(*) FROM SHOPFLOW_DB.SHOPFLOW_RAW.RAW_CUSTOMERS)            +
        (SELECT COUNT(*) FROM SHOPFLOW_DB.SHOPFLOW_RAW.RAW_SELLERS)              +
        (SELECT COUNT(*) FROM SHOPFLOW_DB.SHOPFLOW_RAW.RAW_PRODUCTS)             +
        (SELECT COUNT(*) FROM SHOPFLOW_DB.SHOPFLOW_RAW.RAW_PAYMENTS)             +
        (SELECT COUNT(*) FROM SHOPFLOW_DB.SHOPFLOW_RAW.RAW_REVIEWS)              +
        (SELECT COUNT(*) FROM SHOPFLOW_DB.SHOPFLOW_RAW.RAW_GEOLOCATION)          +
        (SELECT COUNT(*) FROM SHOPFLOW_DB.SHOPFLOW_RAW.RAW_CATEGORY_TRANSLATION)
    AS grand_total
""").collect()[0]['GRAND_TOTAL']

print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  SHOPFLOW — PHASE 1 COMPLETE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Total rows in RAW layer : {total:,}
  Tables loaded           : 9
  Database                : SHOPFLOW_DB
  Schema                  : SHOPFLOW_RAW

  Key Snowflake concepts you practised:
    ✅  Snowpark Session — get_active_session()
    ✅  Internal stage — LIST command
    ✅  COPY INTO — FORMAT_NAME, ON_ERROR, PURGE
    ✅  LOAD_HISTORY — INFORMATION_SCHEMA audit view
    ✅  Raw layer design — all VARCHAR, metadata cols
    ✅  Warehouse suspend / resume

  Next — Phase 2: STAGED layer
    → Cast VARCHAR to proper data types
    → Deduplicate with window functions
    → Handle NULLs and validate referential integrity
    → Automate with Streams + Tasks
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")